# 01. ReAct Agent from Scratch (Working CPU Edition)

**Topics covered:** ReAct Pattern · Reasoning + Acting

Part of the [**Agents**](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/) Seriese

FLAN-T5-small **cannot** reliably emit free-form `tool[arg]` strings. It answers with plain text like `17 * 23` instead.

This edition uses a **two-stage ReAct loop** that small seq2seq models *can* follow:

1. **Choose tool** – multiple-choice (A/B/C/D) ← FLAN-T5 is good at this
2. **Fill argument** – short completion with examples
3. **Observe** – run the tool
4. **Finish** – return the observation (or ask the model to phrase it)

You still get a real Thought → Act → Observe loop on Colab CPU.


## 1. ReAct idea (unchanged)

```text
Thought  →  Action (tool + args)  →  Observation  →  …  →  Final answer
```

The *control structure* is ReAct. Only the **how we ask the LLM** changes so tiny models succeed.


## 2. Setup

```bash
pip install transformers accelerate torch
```


In [52]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

MODEL_NAME = "google/flan-t5-base" if DEVICE == "cuda" else "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"Model: {MODEL_NAME}")


Device: cpu


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model: google/flan-t5-small


## 3. Tools


In [53]:
def tool_calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    expr = expression.strip()
    if not expr or not all(c in allowed for c in expr):
        return "Error: invalid expression"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


def tool_convert(query: str) -> str:
    q = query.lower().strip()
    m = re.match(r"([0-9.]+)\s*([a-z]+)\s+to\s+([a-z]+)", q)
    if not m:
        return "Error: use format like '10 km to miles'"
    value, src, dst = float(m.group(1)), m.group(2), m.group(3)
    table = {
        ("km", "miles"): value * 0.621371,
        ("miles", "km"): value * 1.60934,
        ("kg", "lb"): value * 2.20462,
        ("lb", "kg"): value * 0.453592,
        ("c", "f"): value * 9 / 5 + 32,
        ("f", "c"): (value - 32) * 5 / 9,
    }
    if (src, dst) not in table:
        return f"Error: unsupported {src} to {dst}"
    return f"{table[(src, dst)]:.4g} {dst}"


KB = {
    "capital of france": "Paris",
    "capital of japan": "Tokyo",
    "capital of germany": "Berlin",
    "capital of italy": "Rome",
    "inventor of the telephone": "Alexander Graham Bell",
    "speed of light": "299792 km/s",
    "boiling point of water": "100 C",
}


def tool_search(query: str) -> str:
    q = query.lower().strip()
    for k, v in KB.items():
        if all(w in q for w in k.split()) or k in q or q in k:
            return v
    for k, v in KB.items():
        if any(w in q for w in k.split() if len(w) > 3):
            return v
    return "No result found."


TOOLS = {
    "calculator": tool_calculator,
    "convert": tool_convert,
    "search": tool_search,
}

TOOL_LIST = ["calculator", "convert", "search", "finish"]
print("Tools:", TOOL_LIST)


Tools: ['calculator', 'convert', 'search', 'finish']


## 4. LLM helper


In [54]:
def llm(prompt: str, max_new_tokens: int = 32) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

print(llm("Translate to French: Hello"))


Hello, en anglais.


## 5. Stage A – Choose the tool (multiple choice)

FLAN-T5 is reliable when the answer is a letter or a single word from a fixed list.


In [55]:
def choose_tool(question: str, last_observation: str | None = None) -> str:
    """Return one of: calculator, convert, search, finish."""
    obs_line = f"Last observation: {last_observation}\n" if last_observation else ""
    prompt = f"""Pick the single best tool for the question.
Options:
A) calculator  (math expressions like 17*23)
B) convert     (unit conversion like 10 km to miles)
C) search      (facts like capital of Japan)
D) finish      (we already have the answer from an observation)

{obs_line}Question: {question}
Answer with only one letter A, B, C, or D:"""
    raw = llm(prompt, max_new_tokens=8).upper()
    letter = None
    for ch in raw:
        if ch in "ABCD":
            letter = ch
            break
    mapping = {"A": "calculator", "B": "convert", "C": "search", "D": "finish"}
    tool = mapping.get(letter)
    if tool is None:
        # keyword fallback so demos never hang
        q = question.lower()
        if any(x in q for x in ["convert", "km", "miles", "kg", "lb", " c ", " f ", "celsius", "fahrenheit"]):
            tool = "convert"
        elif any(x in q for x in ["capital", "inventor", "who ", "what is the", "speed of", "boiling" , "light", "invent" ]):
            tool = "search"
        elif any(ch.isdigit() for ch in q) and any(op in q for op in ["+", "-", "*", "/", "x", "times", "plus"]):
            tool = "calculator"
        elif last_observation and not last_observation.lower().startswith("error"):
            tool = "finish"
        else:
            tool = "search"
    return tool


for q in ["What is 17 * 23?", "Convert 10 km to miles", "Capital of Japan?", "Thanks"]:
    print(q, "→", choose_tool(q))


What is 17 * 23? → calculator
Convert 10 km to miles → convert
Capital of Japan? → search
Thanks → calculator


## 6. Stage B – Extract the tool argument


In [56]:
def extract_argument(tool: str, question: str) -> str:
    if tool == "finish":
        return ""
    if tool == "calculator":
        prompt = f"""Extract only the math expression from the question.
Examples:
Question: What is 17 * 23?
Expression: 17*23
Question: Compute (2+3)*4
Expression: (2+3)*4

Question: {question}
Expression:"""
        expr = llm(prompt, max_new_tokens=24)
        # keep only safe characters
        expr = "".join(c for c in expr if c in "0123456789+-*/(). ")
        return expr.strip() or question

    if tool == "convert":
        prompt = f"""Rewrite as: <number> <unit> to <unit>
Examples:
Question: Convert 10 km to miles
Output: 10 km to miles
Question: How many lb in 5 kg?
Output: 5 kg to lb

Question: {question}
Output:"""
        return llm(prompt, max_new_tokens=24).strip().lower()

    # search
    prompt = f"""Extract a short search query (a few words).
Examples:
Question: What is the capital of Japan?
Query: capital of japan
Question: Who invented the telephone?
Query: inventor of the telephone

Question: {question}
Query:"""
    return llm(prompt, max_new_tokens=24).strip().lower()


print(extract_argument("calculator", "What is 17 * 23?"))
print(extract_argument("search", "What is the capital of Japan?"))


17*23
tokyo


## 7. Full ReAct-style loop (two-stage)


In [57]:
def react_agent(question: str, max_steps: int = 4, verbose: bool = True) -> str:
    last_obs = None

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"--- Step {step} ---")

        # Thought (explicit, for teaching)
        thought = f"I need a tool for: {question}" if last_obs is None else f"I observed '{last_obs}' – decide next action"
        if verbose:
            print(f"Thought: {thought}")

        tool = choose_tool(question, last_observation=last_obs)
        if verbose:
            print(f"Action tool chosen: {tool}")

        if tool == "finish":
            answer = last_obs if last_obs and not last_obs.lower().startswith("error") else question
            if verbose:
                print(f"Action: finish[{answer}]")
                print(f"Final answer: {answer}")
            return answer

        arg = extract_argument(tool, question)
        if verbose:
            print(f"Action: {tool}[{arg}]")

        observation = TOOLS[tool](arg)
        last_obs = observation
        if verbose:
            print(f"Observation: {observation}\n")

        # Most demo questions are solved in one tool call → auto finish next
        if not observation.lower().startswith("error") and observation != "No result found.":
            # One more loop iteration will usually select finish via choose_tool
            continue

    if last_obs and not last_obs.lower().startswith("error"):
        if verbose:
            print(f"(Auto-finish) {last_obs}")
        return last_obs
    return "Could not solve the question."


print("Agent ready.")


Agent ready.


## 8. Run examples

These should complete in 1–2 steps with real tool calls.


In [58]:
print("=== Math ===")
print("→", react_agent("What is 17 * 23?"))
print()


=== Math ===
--- Step 1 ---
Thought: I need a tool for: What is 17 * 23?
Action tool chosen: calculator
Action: calculator[17*23]
Observation: 391

--- Step 2 ---
Thought: I observed '391' – decide next action
Action tool chosen: calculator
Action: calculator[17*23]
Observation: 391

--- Step 3 ---
Thought: I observed '391' – decide next action
Action tool chosen: calculator
Action: calculator[17*23]
Observation: 391

--- Step 4 ---
Thought: I observed '391' – decide next action
Action tool chosen: calculator
Action: calculator[17*23]
Observation: 391

(Auto-finish) 391
→ 391



In [59]:
print("=== Unit conversion ===")
print("→", react_agent("Convert 10 km to miles"))
print()


=== Unit conversion ===
--- Step 1 ---
Thought: I need a tool for: Convert 10 km to miles
Action tool chosen: convert
Action: convert[10 km to miles]
Observation: 6.214 miles

--- Step 2 ---
Thought: I observed '6.214 miles' – decide next action
Action tool chosen: convert
Action: convert[10 km to miles]
Observation: 6.214 miles

--- Step 3 ---
Thought: I observed '6.214 miles' – decide next action
Action tool chosen: convert
Action: convert[10 km to miles]
Observation: 6.214 miles

--- Step 4 ---
Thought: I observed '6.214 miles' – decide next action
Action tool chosen: convert
Action: convert[10 km to miles]
Observation: 6.214 miles

(Auto-finish) 6.214 miles
→ 6.214 miles



In [60]:
print("=== Fact search ===")
print("→", react_agent("What is the capital of japan?")) #japan #Knowledge base infomation matters alot
print()


=== Fact search ===
--- Step 1 ---
Thought: I need a tool for: What is the capital of japan?
Action tool chosen: search
Action: search[japan]
Observation: Tokyo

--- Step 2 ---
Thought: I observed 'Tokyo' – decide next action
Action tool chosen: search
Action: search[japan]
Observation: Tokyo

--- Step 3 ---
Thought: I observed 'Tokyo' – decide next action
Action tool chosen: search
Action: search[japan]
Observation: Tokyo

--- Step 4 ---
Thought: I observed 'Tokyo' – decide next action
Action tool chosen: search
Action: search[japan]
Observation: Tokyo

(Auto-finish) Tokyo
→ Tokyo



In [62]:
print("=== Another fact ===")
print("→", react_agent("Who invented the telephone ?")) #Who invented the telephone? boiling point of water
print()

print("=== Test Another fact ===")
print("→", react_agent("i am the telephone")) #Who invented the telephone? boiling point of water
print()

print("=== Test Another fact ===")
print("→", react_agent("Hobs invented the telephone.")) #Who invented the telephone? boiling point of water
print()

=== Another fact ===
--- Step 1 ---
Thought: I need a tool for: Who invented the telephone ?
Action tool chosen: calculator
Action: calculator[.]
Observation: Error: invalid syntax (<string>, line 1)

--- Step 2 ---
Thought: I observed 'Error: invalid syntax (<string>, line 1)' – decide next action
Action tool chosen: calculator
Action: calculator[.]
Observation: Error: invalid syntax (<string>, line 1)

--- Step 3 ---
Thought: I observed 'Error: invalid syntax (<string>, line 1)' – decide next action
Action tool chosen: calculator
Action: calculator[.]
Observation: Error: invalid syntax (<string>, line 1)

--- Step 4 ---
Thought: I observed 'Error: invalid syntax (<string>, line 1)' – decide next action
Action tool chosen: calculator
Action: calculator[.]
Observation: Error: invalid syntax (<string>, line 1)

→ Could not solve the question.

=== Test Another fact ===
--- Step 1 ---
Thought: I need a tool for: i am the telephone
Action tool chosen: calculator
Action: calculator[3 * 4]


## 9. Why this works on FLAN-T5-small

| Approach | Result on tiny models |
|----------|------------------------|
| Free-form `Action: calculator[17*23]` | ❌ Model writes `17 * 23` |
| Constrained `tool[arg]` only | ❌ Still drifts (`action[1]`) |
| **Multiple-choice tool + separate arg extraction** | ✅ Reliable |

You still implement the ReAct **control flow** (thought → act → observe → finish).  
Only the interface to the weak LLM is adapted.

With a stronger chat model you can switch back to a single free-form ReAct prompt or JSON tool calls.


## 10. Why Small Models Struggle (and how to improve)

Check this notebook [`01_react_agent_from_scratch.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/01_react_agent_from_scratch.ipynb)

## 11. Summary

| Stage | What happens |
|-------|----------------|
| Choose tool | A/B/C/D classification (+ keyword fallback) |
| Extract arg | Short targeted completion |
| Run tool | Plain Python function |
| Finish | Return observation as answer |

```python
tool = choose_tool(question, last_obs)
if tool == "finish":
    return last_obs
arg = extract_argument(tool, question)
last_obs = TOOLS[tool](arg)
```

---

**Next notebook:** [`02_tool_calling_agent.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/02_tool_calling_agent.ipynb.ipynb)
Tool Calling · Function Calling · Memory


---
**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities